# Task 2 classifier baseline
Run setup, then classifier training. Settings are provisional; Optuna and specialists remain pending. Validation only, no official test evaluation.


In [ ]:
import torch
print(torch.__version__)
assert torch.cuda.is_available(), "Select a GPU runtime first"
print(torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# Before starting a new experiment, fetch the implementation you pushed.
# When resuming, keep the source revision used by your checkpoint.
subprocess.run(['git', 'pull', '--ff-only'], check=True)
assert (repo / 'training/train_classifier.py').is_file(), 'Push the new files first'
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)


If Colab requests a session restart after installation, restart and rerun setup. Keep the original source revision when resuming; provenance checks reject changed training sources.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/GenAI_A1')
os.chdir(repo)
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)
output = Path('/content/drive/MyDrive/GenAI_A1/task2-classifier')


## Train or resume
Ten epochs. Each training image produces four classes per epoch: 11,776 examples, unlike Task 1. Batches are exactly balanced. Best checkpoint uses macro F1. The cell resumes automatically if last.pt exists; keep configuration/source unchanged.


In [ ]:
command = [sys.executable, '-m', 'training.train_classifier', '--device', 'cuda',
           '--max-hours', '4', '--output-dir', str(output)]
if (output / 'last.pt').exists():
    command += ['--resume', str(output / 'last.pt')]
subprocess.run(command, check=True)


In [ ]:
import json
report = json.loads((output / 'best_validation.json').read_text())
print('Best epoch:', report['epoch'])
for key in ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']:
    print(key, report[key])
print('Class order:', report['class_order'])
print('Normalized confusion (true rows, predicted columns):')
for row in report['normalized_confusion']: print(row)


Share history.json and best_validation.json from MyDrive/GenAI_A1/task2-classifier. If no epoch completed, resume training before running the report cell. Preserve the entire output folder.
